In [43]:
'''
Imp algo in document clustering
but it compramize cohesivity

silhouette_score =  How well-separated and distinct the clusters are from each other.
                    Higher is better (ranges from -1 to +1).
''' 


'\nImp algo in document clustering\nbut it compramize cohesivity\n\nsilhouette_score =  How well-separated and distinct the clusters are from each other.\n                    Higher is better (ranges from -1 to +1).\n'

In [4]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.cluster import KMeans ,DBSCAN
from sklearn.preprocessing import StandardScaler
import os
from sklearn.metrics import silhouette_score
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")

os.chdir(r'E:\Backup Files Abhinav_Satyam\Practical Machine Learning\Sanjay Sir Git Repo\Datasets')

In [5]:
milk = pd.read_csv('milk.csv', index_col = 0)
milk

,water,protein,fat,lactose,ash
tAnimal,,,,,
HORSE,90.1,2.6,1.0,6.9,0.35
ORANGUTAN,88.5,1.4,3.5,6.0,0.24
MONKEY,88.4,2.2,2.7,6.4,0.18
DONKEY,90.3,1.7,1.4,6.2,0.40
HIPPO,90.4,0.6,4.5,4.4,0.10
CAMEL,87.7,3.5,3.4,4.8,0.71
BISON,86.9,4.8,1.7,5.7,0.90
BUFFALO,82.1,5.9,7.9,4.7,0.78
GUINEA PIG,81.9,7.4,7.2,2.7,0.85


In [21]:
scaler = StandardScaler().set_output(transform='pandas')
scl_milk = scaler.fit_transform(milk)

scl_milk

,water,protein,fat,lactose,ash
tAnimal,,,,,
HORSE,0.948806,-1.009291,-0.903208,1.542217,-1.037554
ORANGUTAN,0.821407,-1.344603,-0.660619,1.040773,-1.259945
MONKEY,0.813445,-1.121062,-0.738247,1.263637,-1.381249
DONKEY,0.964731,-1.260775,-0.864394,1.152205,-0.936467
HIPPO,0.972694,-1.568145,-0.563583,0.149319,-1.542988
CAMEL,0.757707,-0.757806,-0.670322,0.372182,-0.309730
BISON,0.694008,-0.394551,-0.835283,0.873626,0.074400
BUFFALO,0.311810,-0.087181,-0.233662,0.316466,-0.168208
GUINEA PIG,0.295885,0.331959,-0.301587,-0.797852,-0.026687


In [34]:
dbs = DBSCAN(eps=1.5 , min_samples=2)  
dbs.fit(scl_milk)
# dbs.labels_   -1 indicates outliers
inliers_obs = scl_milk[dbs.labels_ != -1]  # removing outliers
inliers_lbls = dbs.labels_[dbs.labels_ != -1]  # removing outliers
print(inliers_obs)
print('\n\n')
print(inliers_lbls)

               water   protein       fat   lactose       ash
tAnimal                                                     
HORSE       0.948806 -1.009291 -0.903208  1.542217 -1.037554
ORANGUTAN   0.821407 -1.344603 -0.660619  1.040773 -1.259945
MONKEY      0.813445 -1.121062 -0.738247  1.263637 -1.381249
DONKEY      0.964731 -1.260775 -0.864394  1.152205 -0.936467
HIPPO       0.972694 -1.568145 -0.563583  0.149319 -1.542988
CAMEL       0.757707 -0.757806 -0.670322  0.372182 -0.309730
BISON       0.694008 -0.394551 -0.835283  0.873626  0.074400
BUFFALO     0.311810 -0.087181 -0.233662  0.316466 -0.168208
GUINEA PIG  0.295885  0.331959 -0.301587 -0.797852 -0.026687
CAT         0.271998  1.086413 -0.388919  0.149319 -0.228860
FOX         0.271998  0.108418 -0.427733  0.427898  0.135052
LLAMA       0.662158 -0.646036 -0.689729  0.817910 -0.127774
MULE        0.940844 -1.176947 -0.825579  0.762194 -0.794946
PIG         0.367547  0.248131 -0.505362 -0.240693  0.478747
ZEBRA       0.638270 -0.

In [20]:
silhouette_score(inliers_obs, inliers_lbls)

0.5528890106341011

## Trying with different epsilon values and min samples

In [41]:
epsilons= np.linspace(0.1, 3 , 20)  # radius of epsilons
minpts = [2,3,4,5]
scores = []
for e in tqdm(epsilons):
    for p in minpts:
        dbs = DBSCAN(eps=e , min_samples=p)  
        dbs.fit(scl_milk) 
        
        inliers_obs = scl_milk[dbs.labels_ != -1]  # removing outliers
        inliers_lbls = dbs.labels_[dbs.labels_ != -1]  # removing outliers

        if len(set(inliers_lbls))>1:     # silhouette_score requires atlest 2 clusters to compare
            scores.append([e, p , silhouette_score(inliers_obs, inliers_lbls)])
            
df_scores = pd.DataFrame(scores , columns = ['eps' , 'min_samples', 'silhouette_score'])

df_scores.sort_values('silhouette_score', ascending=False)

100%|██████████| 20/20 [00:00<00:00, 80.26it/s]


,eps,min_samples,silhouette_score
8,0.863158,3,0.657551
0,0.405263,2,0.651894
2,0.557895,2,0.593446
10,1.168421,2,0.552889
12,1.473684,2,0.552889
11,1.321053,2,0.552889
4,0.557895,4,0.551975
9,1.015789,4,0.539851
1,0.405263,3,0.538518
3,0.557895,3,0.534443


In [44]:
# comparing witg KMeans the best score of the DBSCAN is best


# HDBSCAN

In [45]:
from sklearn.cluster import HDBSCAN

In [52]:
hdbs = HDBSCAN(min_cluster_size=3)  
hdbs.fit(scl_milk)
print('labels = ',hdbs.labels_)

inliers_obs = scl_milk[hdbs.labels_ != -1]    # removing outliers
inliers_lbls = hdbs.labels_[hdbs.labels_ != -1]  # removing outliers
print('\nsilhouette_score : ', silhouette_score(inliers_obs, inliers_lbls))

labels =  [ 2  2  2  2 -1  3 -1  1 -1 -1  1  3  2  1  3  1  0 -1 -1  0  0  0  0 -1
 -1]

silhouette_score :  0.5032777845817115
